# Rerun W&B Run Plot

Focused notebook backed by `wandb_metrics.py` for GINE best_00 rerun comparisons, A0 rerun comparisons, and paper-baseline comparisons.

For single-panel multi-curve plots, edit the `*_RUN_SPECS` cell directly: use `name` for one exact run, `prefix` for all runs starting with a prefix, `regex` for a regular expression, and `label` for the legend name shown on the plot.


In [18]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)
print("wandb_metrics:", wm.__file__)


wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py


## Load Selected W&B Histories


In [19]:
data = wm.load_wandb_data()

runs_df = data.runs_df
history_df = data.history_df

Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 308 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    203
crashed     100
killed        5
History artifact setup: local_only=True, runs_df=308
[ 1/308] loading artifact cache: a0_aib_00_flat_local_t020_s0
    loaded 359 rows, 148 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_cpu/runs/a0_aib_00_flat_local_t020_s0__MAPPO_bus14_T_0_0__I__1782784223_20854/history.parquet in 0.2s
[ 2/308] loading artifact cache: a0_aib_00_flat_local_t020_s0
    loaded 356 rows, 148 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_gpu/runs

## GINE Rerun Comparisons

`gine_rerun_result` is built from the editable `GINE_RERUN_SUBPLOTS` dictionary below. The default baseline is now `rerun_a0_opt_s*`, displayed as `A0_baseline`. The first subplot compares this new baseline against the ancient GINE `best_00` baseline.


Assuming plot baseline = `A0_baseline` (`rerun_a0_opt_s*`). The GINE rows below are read from `Topology_Task/configs/gine_s0_s1_s2`; the A0 baseline row is included because it is the reference curve used in `gine_rerun_result`.

|Displayed name | Actor/Critic encoder | GNN hidden x layers | Actor/Critic MLP | Concat flat | Edge pre-enc. | Entropy | Init action-0 | Optimized critic | Difference vs `A0_baseline` |
|---|---|---|---|---|---|---:|---:|---|---|
| `A0_baseline` | `mlp` / `mlp` | `-` | `[256,256,256]` / `[256,256,256]` | `-` | `-` | `0.01` | `0.0` | default | Plot baseline: flat A0 MLP policy. |
| `ancient_GINE_baseline` | `gnn` / `gnn` | `128 x 2` | `[256,256,256]` / `[256,256,256]` | `true` | `true` | `0.02` | `0.7` | `false` | Former GINE baseline: heavy shared GINE actor, GNN critic, flat-observation concatenation, edge pre-encoder, legacy critic update. |
| `10_light_GINE_MLP_critic` | `gnn` / `mlp` | `64 x 1` | `[128,128]` / `[128,128]` | `true` | `false` | `0.02` | `0.7` | `true` | Light GINE actor with MLP critic, smaller heads, no edge pre-encoder, optimized critic update. |
| `11_legacy_init_bias_0.0` | `gnn` / `gnn` | `128 x 2` | `[256,256,256]` / `[256,256,256]` | `true` | `true` | `0.02` | `0.0` | `false` | Same heavy/legacy GINE family as `best_00`, but removes the initial action-0 bias. |
| `12_optcritic_init_bias_0.0` | `gnn` / `gnn` | `128 x 2` | `[256,256,256]` / `[256,256,256]` | `true` | `true` | `0.02` | `0.0` | `true` | Heavy GINE with no initial action-0 bias and optimized critic update. |
| `13_GINE_A0_GNN_critic` | `gnn` / `gnn` | `128 x 2` | `[256,256,256]` / `[256,256,256]` | `true` | `true` | `0.01` | `0.0` | `true` | A0-style heavy GINE: lower entropy, no initial action-0 bias, optimized critic update. |
| `14_light_GINE_A0_MLP_critic` | `gnn` / `mlp` | `64 x 1` | `[128,128]` / `[128,128]` | `false` | `false` | `0.01` | `0.0` | `true` | A0-style light GINE: MLP critic, no flat-observation concatenation, no edge pre-encoder, lower entropy. |

Shared unless noted for GINE rows: `env_id = bus14`, seeds `0/1/2`, `n_envs = 20`, `n_steps = 2000`, `norm_reward = true`, `total_timesteps = 15000000`, `eval_freq = 80000`, split chronics with `test_chronics_pct = 0.2`, shared actor GNN, and node-id embeddings enabled.

Edit the cell below to choose, for each subplot, which runs are plotted and what label is displayed in the legend.

Supported selectors are `name`, `prefix`, `regex`, `contains`, `runs`, and `run_dir`. The legend text is controlled by `label`.



In [20]:

GINE_RERUN_A0_BASELINE_SPEC = {
    "prefix": "rerun_a0_opt_s",
    "label": "A0_baseline",
    "color": "#1f77b4",
    "width": 4,
}
GINE_RERUN_ANCIENT_BASELINE_SPEC = {
    "prefix": "best_00_shared_actor_gnn_gine_a4_concat_flat_critic_gnn_legacy_update_s",
    "label": "ancient_GINE_baseline",
    "color": "#7f7f7f",
    "dash": "dash",
    "width": 3,
}

GINE_RERUN_SUBPLOTS = {
    "A0 baseline vs ancient GINE baseline": [
        GINE_RERUN_A0_BASELINE_SPEC,
        GINE_RERUN_ANCIENT_BASELINE_SPEC,
    ],
    "A0 baseline vs 10 light GINE MLP critic": [
        GINE_RERUN_A0_BASELINE_SPEC,
        {
            "prefix": "best_10_shared_actor_gnn_light_gine_a4_concat_flat_critic_mlp_optcritic_s",
            "label": "10_light_GINE_MLP_critic",
            "color": "#ff7f0e",
        },
    ],
    "A0 baseline vs 11 legacy init bias 0.0": [
        GINE_RERUN_A0_BASELINE_SPEC,
        {
            "prefix": "best_11_shared_actor_gnn_gine_a4_concat_flat_critic_gnn_legacy_update_initbias0_s",
            "label": "11_legacy_init_bias_0.0",
            "color": "#ff7f0e",
        },
    ],
    "A0 baseline vs 12 optcritic init bias 0.0": [
        GINE_RERUN_A0_BASELINE_SPEC,
        {
            "prefix": "best_12_shared_actor_gnn_gine_a4_concat_flat_critic_gnn_opt_initbias0_s",
            "label": "12_optcritic_init_bias_0.0",
            "color": "#ff7f0e",
        },
    ],
    "A0 baseline vs 13 GINE A0 GNN critic": [
        GINE_RERUN_A0_BASELINE_SPEC,
        {
            "prefix": "best_13_shared_actor_gnn_gine_a0_concat_flat_critic_gnn_opt_s",
            "label": "13_GINE_A0_GNN_critic",
            "color": "#ff7f0e",
        },
    ],
    "A0 baseline vs 14 light GINE A0 MLP critic": [
        GINE_RERUN_A0_BASELINE_SPEC,
        {
            "prefix": "best_14_shared_actor_gnn_light_gine_a0_no_concat_flat_critic_mlp_optcritic_s",
            "label": "14_light_GINE_A0_MLP_critic",
            "color": "#ff7f0e",
        },
    ],
}

GINE_RERUN_MEAN_GROUPS = {
    title: wm.resolve_named_plot_specs(run_specs, history=history_df)
    for title, run_specs in GINE_RERUN_SUBPLOTS.items()
}
GINE_RERUN_MEAN_GROUPS = {
    title: specs
    for title, specs in GINE_RERUN_MEAN_GROUPS.items()
    if len(specs) >= 2
}

gine_rerun_result = {
    "mean_groups": GINE_RERUN_MEAN_GROUPS,
    "fig": wm.plot_run_mean_groups(
        GINE_RERUN_MEAN_GROUPS,
        split="test",
        smooth=5,
        title="GINE reruns: editable comparisons with A0_baseline",
        ncols=2,
        subplot_height=400,
        width=1500,
        y_range=[0, 105],
        show_members=True,
        show_std=True,
        save_name="rerun_gine_a0_baseline_editable_comparisons",
        history=history_df,
    ),
}
gine_rerun_result["fig"]


Plot source folders used to build curves: 2 folder(s), 21 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_test_rerun (3 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/gine_s0_s1_s2 (18 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/rerun_gine_a0_baseline_editable_comparisons.html


## A0 Core Rerun Comparisons

- `rerun_a0_opt`      = baseline
- `rerun_a0_nonopt`   = baseline without optimized critic updates
- `rerun_a01_opt`     = baseline with smaller MLP
- `rerun_a02_opt`     = baseline without reward normalization
- `old_a0_nonopt_noval20` = baseline without optimized critic updates and init do nothing bias = 0.3

`a0_core_pairwise_result` is built from the editable `A0_CORE_PAIRWISE_SUBPLOTS` dictionary below. The baseline is still `rerun_a0_opt_s*`; only the run choices and legend labels are now editable.


Assuming baseline = `rerun_a0_opt`.

| Run family | Actor/Critic layers | Reward norm | Init do-nothing prob | Optimized critic updates | Difference vs `rerun_a0_opt` |
|---|---|---|---:|---|---|
| `rerun_a0_opt` | `[256,256,256]` / `[256,256,256]` | `true` | `0.0` | `true` default | Baseline |
| `rerun_a0_nonopt` | `[256,256,256]` / `[256,256,256]` | `true` | `0.0` | `false` | disables optimized critic updates |
| `rerun_a01` | `[256,256]` / `[256,256]` | `true` | `0.0` | `true` default | smaller MLP actor/critic |
| `rerun_a02` | `[256,256,256]` / `[256,256,256]` | `false` | `0.0` | `true` default | disables reward normalization |
| `old_a0_nonopt_noval20` | `[256,256,256]` / `[256,256,256]` | `true` | `0.3` | `false` | bias `0.3`, non-optimized critic |
| `rerun_bias_05` | `[256,256,256]` / `[256,256,256]` | `true` | `0.5` | `true` default | bias `0.5` |
| `rerun_bias_07` | `[256,256,256]` / `[256,256,256]` | `true` | `0.7` | `true` default | bias `0.7` |

In [21]:

A0_CORE_PAIRWISE_BASELINE_SPEC = {
    "prefix": "rerun_a0_opt_s",
    "label": "rerun_a0_opt",
    "color": "#1f77b4",
    "width": 4,
}

A0_CORE_PAIRWISE_SUBPLOTS = {
    "rerun_a0_opt vs rerun_a0_nonopt": [
        A0_CORE_PAIRWISE_BASELINE_SPEC,
        {
            "prefix": "rerun_a0_nonopt_s",
            "label": "rerun_a0_nonopt",
            "color": "#ff7f0e",
        },
    ],
    "rerun_a0_opt vs rerun_a01": [
        A0_CORE_PAIRWISE_BASELINE_SPEC,
        {
            "prefix": "rerun_a01_opt_s",
            "label": "rerun_a01",
            "color": "#2ca02c",
        },
    ],
    "rerun_a0_opt vs rerun_a02": [
        A0_CORE_PAIRWISE_BASELINE_SPEC,
        {
            "prefix": "rerun_a02_opt_s",
            "label": "rerun_a02",
            "color": "#d62728",
        },
    ],
    "rerun_a0_opt vs old_a0_nonopt_noval20": [
        A0_CORE_PAIRWISE_BASELINE_SPEC,
        {
            "regex": r"^noval20_mlp_a1_no_entropy_decay_nonopt_s[0-2]_det$",
            "label": "old_a0_nonopt_noval20",
            "color": "#9467bd",
        },
    ],
}

A0_CORE_PAIRWISE_MEAN_GROUPS = {
    title: wm.resolve_named_plot_specs(run_specs, history=history_df)
    for title, run_specs in A0_CORE_PAIRWISE_SUBPLOTS.items()
}
A0_CORE_PAIRWISE_MEAN_GROUPS = {
    title: specs
    for title, specs in A0_CORE_PAIRWISE_MEAN_GROUPS.items()
    if len(specs) >= 2
}

a0_core_pairwise_result = {
    "mean_groups": A0_CORE_PAIRWISE_MEAN_GROUPS,
    "fig": wm.plot_run_mean_groups(
        A0_CORE_PAIRWISE_MEAN_GROUPS,
        split="test",
        smooth=5,
        title="A0 core reruns: editable pairwise comparisons",
        ncols=2,
        subplot_height=430,
        width=1500,
        y_range=[0, 105],
        show_members=True,
        show_std=True,
        save_name="rerun_a0_core_pairwise_editable_comparisons",
        history=history_df,
    ),
}
a0_core_pairwise_result["fig"]


Missing runs matching regex '^noval20_mlp_a1_no_entropy_decay_nonopt_s[0-2]_det$'
Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_test_rerun (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/rerun_a0_core_pairwise_editable_comparisons.html


In [22]:

A0_CORE_ALL_RUN_SPECS = [
    {"prefix": "rerun_a0_opt_s", "label": "rerun_a0_opt", "color": "#1f77b4", "width": 4},
    {"prefix": "rerun_a0_nonopt_s", "label": "rerun_a0_nonopt", "color": "#ff7f0e"},
    {"prefix": "rerun_a01_opt_s", "label": "rerun_a01", "color": "#2ca02c"},
    {"prefix": "rerun_a02_opt_s", "label": "rerun_a02", "color": "#d62728"},
    {
        "regex": r"^noval20_mlp_a1_no_entropy_decay_nonopt_s[0-2]_det$",
        "label": "old_a0_nonopt_noval20",
        "color": "#9467bd",
    },
]

a0_core_all_result = wm.plot_named_run_means(
    A0_CORE_ALL_RUN_SPECS,
    title="A0 core reruns: editable single-panel comparison",
    split="test",
    smooth=5,
    show_members=True,
    show_std=True,
    save_name="rerun_a0_core_all_curves_editable",
)
a0_core_all_result["fig"]


Missing runs matching regex '^noval20_mlp_a1_no_entropy_decay_nonopt_s[0-2]_det$'
Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_test_rerun (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/rerun_a0_core_all_curves_editable.html


## A0 Bias Comparison


In [23]:

A0_BIAS_RUN_SPECS = [
    {"prefix": "rerun_a0_opt_s", "label": "rerun_a0_opt", "color": "#1f77b4", "width": 4},
    {"name": "rerun_bias_05", "label": "rerun_bias_05", "color": "#ff7f0e"},
    {"name": "rerun_bias_07", "label": "rerun_bias_07", "color": "#d62728"},
]

a0_bias_result = wm.plot_named_run_means(
    A0_BIAS_RUN_SPECS,
    title="A0 baseline vs init-bias controls",
    split="test",
    smooth=5,
    show_members=True,
    show_std=True,
    save_name="rerun_a0_opt_vs_bias05_bias07_editable",
)
a0_bias_result["fig"]


Plot source folders used to build curves: 1 folder(s), 5 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_test_rerun (5 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/rerun_a0_opt_vs_bias05_bias07_editable.html


## Paper Baseline vs A0 Baselines

Edit `PAPER_BASELINE_RUN_SPECS` to choose which downloaded runs appear and how they are named in the legend. This cell intentionally selects only `paper_mappo_discrete_baseline_thread16`, so `_thread1` is not plotted or shown in the legend.


In [24]:

PAPER_BASELINE_RUN_SPECS = [
    {
        "name": "paper_mappo_discrete_baseline_thread16",
        "label": "paper_mappo_discrete_baseline",
        "color": "#111111",
        "width": 4,
    },
    {"prefix": "rerun_a0_opt_s", "label": "A0_baseline", "color": "#1f77b4", "width": 4},
    {"name": "rerun_bias_05", "label": "A0_baseline_bias05", "color": "#ff7f0e"},
    {"name": "rerun_bias_07", "label": "A0_baseline_bias07", "color": "#d62728"},
]

paper_vs_a0_baseline_result = wm.plot_named_run_means(
    PAPER_BASELINE_RUN_SPECS,
    title="Paper MAPPO discrete baseline vs A0 baselines",
    split="test",
    smooth=5,
    show_members=True,
    show_std=True,
    save_name="paper_mappo_discrete_baseline_vs_a0_baselines",
)
paper_vs_a0_baseline_result["fig"]


Plot source folders used to build curves: 2 folder(s), 6 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_test_rerun (5 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/mappo_discrete_baseline (1 run)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/paper_mappo_discrete_baseline_vs_a0_baselines.html
